In [16]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

def run_maximal_gbr_pipeline(input_filename, output_filename):
    # 1. --- LOAD DATABASE ---
    df = pd.read_csv(input_filename)
    print(f"📖 Loaded '{input_filename}' with {len(df)} players.")
    
    # 2. --- TARGET SETUP (Inverting ranks so higher target value = better player) ---
    rank_cols = ['Modern_Consensus_Rank', 'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']
    valid_ranks = [c for c in rank_cols if c in df.columns]
        
    df['Consensus_Target'] = -df[valid_ranks].mean(axis=1)

    # 4. --- AUTOMATED METRIC SELECTION ---
    # Drops identifying info, raw targets, and pre-computed models to find pure analytics columns
    exclude_cols = [
        'Player_Name', 'Era_Label', 'Archetype_Label', 'Era_ID', 'Archetype_ID',
        'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank', 
        'Modern_Athletic_Raw_Rank', 'Modern_Yahoo_Raw_Rank', 'Modern_BR_Raw_Rank',
        'Modern_Consensus_Rank', 'Scratch_DL_Score', 'Scratch_DL_Rank',
        'Consensus_Target', 'GBR_All_Prediction', 'Maximal_Greatness_Index', 'Maximal_ML_Rank'
    ]
    all_metrics = [col for col in df.columns if col in df.select_dtypes(include=[np.number]).columns and col not in exclude_cols]

    # 5. --- DATA CLEANING PIPELINE ---
    df_clean = df.copy()
    for col in all_metrics:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        # Robust fallback: fill missing data with column medians to prevent training corruption
        df_clean[col] = df_clean[col].fillna(df_clean[col].median() if not df_clean[col].isna().all() else 0.0)

    # 6. --- GRADIENT BOOSTING MACHINE FIT ---
    X = df_clean[all_metrics]
    y = df_clean['Consensus_Target']

    gbr_all = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
    gbr_all.fit(X, y)

    # 7. --- MAP TO 0-100 GLOBAL INDEX SCALE ---
    df_clean['GBR_All_Prediction'] = gbr_all.predict(X)
    min_val = df_clean['GBR_All_Prediction'].min()
    max_val = df_clean['GBR_All_Prediction'].max()
    
    if max_val != min_val:
        df_clean['Maximal_Greatness_Index'] = ((df_clean['GBR_All_Prediction'] - min_val) / (max_val - min_val)) * 100
    else:
        df_clean['Maximal_Greatness_Index'] = 100.0

    # 8. --- COMPUTE ORDERED DENSE RANKINGS ---
    df_clean['Maximal_ML_Rank'] = df_clean['Maximal_Greatness_Index'].rank(ascending=False, method='min').astype(int)
    df_final = df_clean.sort_values('Maximal_ML_Rank')

    # 9. --- EXPORT MASTER RESULT SYSTEM ---
    df_final.to_csv(output_filename, index=False)
    print(f"💾 Success! Complete maximal GBR model database exported to '{output_filename}'\n")

    # 10. --- LEADERBOARD & IMPORTANT FEATURE SPLITS PRINTS ---
    print("🏆 ==================== TOP 10 MAXIMAL GBR INDEX ====================")
    for _, row in df_final.head(10).iterrows():
        print(f" Rank {int(row['Maximal_ML_Rank']):<2} | {row['Player_Name']:<22} | Score: {row['Maximal_Greatness_Index']:.2f}")
        
    print("\n🌲 ==================== TOP 10 FEATURE IMPORTANCES ====================")
    importances = pd.DataFrame({
        'Metric': all_metrics,
        'Importance': gbr_all.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    for rank, (_, row) in enumerate(importances.head(10).iterrows(), 1):
        print(f" {rank:<2} | {row['Metric']:<22} | Split Relative Weight: {row['Importance']*100:.2f}%")

if __name__ == '__main__':
    # Set target structures and call pipeline
    input_db = os.path.join('..', 'nba_modern_filtered_master.csv')
    output_db = 'Gradient_Boosted_Metrics_Values.csv'
    run_maximal_gbr_pipeline(input_db, output_db)

📖 Loaded '..\nba_modern_filtered_master.csv' with 88 players.
💾 Success! Complete maximal GBR model database exported to 'Gradient_Boosted_Metrics_Values.csv'

🏆 ==================== TOP 10 MAXIMAL GBR INDEX ====================
 Rank 1  | Michael Jordan         | Score: 100.00
 Rank 2  | LeBron James           | Score: 99.25
 Rank 3  | Kareem Abdul-Jabbar    | Score: 97.75
 Rank 4  | Magic Johnson          | Score: 96.13
 Rank 5  | Tim Duncan             | Score: 95.27
 Rank 6  | Shaquille O'Neal       | Score: 94.20
 Rank 7  | Larry Bird             | Score: 93.65
 Rank 8  | Kobe Bryant            | Score: 92.26
 Rank 9  | Stephen Curry          | Score: 89.68
 Rank 10 | Kevin Durant           | Score: 89.66

🌲 ==================== TOP 10 FEATURE IMPORTANCES ====================
 1  | All_NBA_Teams          | Split Relative Weight: 69.49%
 2  | Championships          | Split Relative Weight: 8.85%
 3  | All_NBA_PS             | Split Relative Weight: 4.21%
 4  | Playoff_Games        

In [17]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load files
master_file = os.path.join('..', 'nba_modern_filtered_master.csv')
gb_file = 'Gradient_Boosted_Advanced_Metrics.csv'

df_master = pd.read_csv(master_file)
df_gb = pd.read_csv(gb_file)

# 2. Get ground truth consensus target
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# 3. Align data
df_eval = pd.merge(df_gb[['Player_Name', 'Maximal_ML_Rank']], df_master[['Player_Name', 'Consensus_Rank']], on='Player_Name')
df_eval['Rank_Delta'] = df_eval['Maximal_ML_Rank'] - df_eval['Consensus_Rank']

# 4. Calculate stats
mae = mean_absolute_error(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank'])
rmse = np.sqrt(mean_squared_error(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank']))
r2 = r2_score(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank'])

print("🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬")
print(f"📈 Model R² Score (Variance Explained): {r2:.4f}")
print(f"🎯 Mean Absolute Error (Average Deviation): {mae:.2f} positions")
print(f"📉 Root Mean Squared Error (Outlier Penalty): {rmse:.2f} positions")
print("-" * 68)

# Show top 3 closest and top 3 furthest
print("\n🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nsmallest(3).index][['Player_Name', 'Consensus_Rank', 'Maximal_ML_Rank', 'Rank_Delta']].to_string(index=False))

print("\n⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nlargest(3).index][['Player_Name', 'Consensus_Rank', 'Maximal_ML_Rank', 'Rank_Delta']].to_string(index=False))
print("=====================================================================")

🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬
📈 Model R² Score (Variance Explained): 0.8155
🎯 Mean Absolute Error (Average Deviation): 11.64 positions
📉 Root Mean Squared Error (Outlier Penalty): 13.26 positions
--------------------------------------------------------------------

🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:
   Player_Name  Consensus_Rank  Maximal_ML_Rank  Rank_Delta
Michael Jordan        1.000000                1    0.000000
  LeBron James        2.000000                2    0.000000
 Dirk Nowitzki       18.333333               18   -0.333333

⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:
  Player_Name  Consensus_Rank  Maximal_ML_Rank  Rank_Delta
   Chris Bosh       82.333333               58  -24.333333
 Alex English       85.333333               63  -22.333333
Manu Ginobili       73.000000               52  -21.000000


In [18]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

def run_maximal_gbr_pipeline(input_filename, output_filename):
    # 1. --- LOAD DATABASE ---
    df = pd.read_csv(input_filename)
    print(f"📖 Loaded '{input_filename}' with {len(df)} players.")
    
    # 2. --- TARGET SETUP (Inverting ranks so higher target value = better player) ---
    rank_cols = ['Modern_Consensus_Rank', 'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']
    valid_ranks = [c for c in rank_cols if c in df.columns]
        
    df['Consensus_Target'] = -df[valid_ranks].mean(axis=1)

    # 4. --- EXPLICIT ADVANCED METRIC SELECTION ---
    # Enforcing the strict whitelist of your 8 advanced efficiency/rate metrics
    all_metrics = [
        'Career_PER', 'Peak_5Yr_PER', 'Playoff_PER',
        'Career_BPM', 'Playoff_BPM', 'Career_TS_Pct', 
        'Career_WS_per_48', 'Playoff_WS_per_48'
    ]
    
    print(f"🌲 Restricting model features strictly to the {len(all_metrics)} defined advanced metrics.")

    # 5. --- DATA CLEANING PIPELINE ---
    df_clean = df.copy()
    for col in all_metrics:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            # Robust fallback: fill missing data with column medians to prevent training corruption
            df_clean[col] = df_clean[col].fillna(df_clean[col].median() if not df_clean[col].isna().all() else 0.0)

    # 6. --- GRADIENT BOOSTING MACHINE FIT ---
    X = df_clean[all_metrics]
    y = df_clean['Consensus_Target']

    gbr_all = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.2, random_state=42)
    gbr_all.fit(X, y)

    # 7. --- MAP TO 0-100 GLOBAL INDEX SCALE ---
    df_clean['GBR_All_Prediction'] = gbr_all.predict(X)
    min_val = df_clean['GBR_All_Prediction'].min()
    max_val = df_clean['GBR_All_Prediction'].max()
    
    if max_val != min_val:
        df_clean['Maximal_Greatness_Index'] = ((df_clean['GBR_All_Prediction'] - min_val) / (max_val - min_val)) * 100
    else:
        df_clean['Maximal_Greatness_Index'] = 100.0

    # 8. --- COMPUTE ORDERED DENSE RANKINGS ---
    df_clean['Maximal_ML_Rank'] = df_clean['Maximal_Greatness_Index'].rank(ascending=False, method='min').astype(int)
    df_final = df_clean.sort_values('Maximal_ML_Rank')

    # 9. --- EXPORT MASTER RESULT SYSTEM ---
    df_final.to_csv(output_filename, index=False)
    print(f"💾 Success! Advanced stats GBR model database exported to '{output_filename}'\n")

    # 10. --- LEADERBOARD & IMPORTANT FEATURE SPLITS PRINTS ---
    print("🏆 ==================== TOP 10 MAXIMAL GBR INDEX ====================")
    for _, row in df_final.head(10).iterrows():
        print(f" Rank {int(row['Maximal_ML_Rank']):<2} | {row['Player_Name']:<22} | Score: {row['Maximal_Greatness_Index']:.2f}")
        
    print("\n🌲 ==================== FEATURE IMPORTANCE BREAKDOWN ====================")
    importances = pd.DataFrame({
        'Metric': all_metrics,
        'Importance': gbr_all.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    for rank, (_, row) in enumerate(importances.iterrows(), 1):
        print(f" {rank:<2} | {row['Metric']:<22} | Split Relative Weight: {row['Importance']*100:.2f}%")

if __name__ == '__main__':
    # Set target structures and call pipeline
    input_db = os.path.join('..', 'nba_modern_filtered_master.csv')
    output_db = 'Gradient_Boosted_Advanced_Metrics.csv'
    run_maximal_gbr_pipeline(input_db, output_db)

📖 Loaded '..\nba_modern_filtered_master.csv' with 88 players.
🌲 Restricting model features strictly to the 8 defined advanced metrics.
💾 Success! Advanced stats GBR model database exported to 'Gradient_Boosted_Advanced_Metrics.csv'

🏆 ==================== TOP 10 MAXIMAL GBR INDEX ====================
 Rank 1  | Michael Jordan         | Score: 100.00
 Rank 2  | LeBron James           | Score: 97.62
 Rank 3  | Magic Johnson          | Score: 95.66
 Rank 4  | Kareem Abdul-Jabbar    | Score: 95.53
 Rank 5  | Tim Duncan             | Score: 93.21
 Rank 6  | Shaquille O'Neal       | Score: 93.20
 Rank 7  | Larry Bird             | Score: 92.49
 Rank 8  | Kobe Bryant            | Score: 89.31
 Rank 9  | Hakeem Olajuwon        | Score: 88.39
 Rank 10 | Stephen Curry          | Score: 88.37

🌲 ==================== FEATURE IMPORTANCE BREAKDOWN ====================
 1  | Career_PER             | Split Relative Weight: 37.93%
 2  | Peak_5Yr_PER           | Split Relative Weight: 24.14%
 3  | Caree

In [19]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load files
master_file = os.path.join('..', 'nba_modern_filtered_master.csv')
gb_file = 'Gradient_Boosted_Advanced_Metrics.csv'

df_master = pd.read_csv(master_file)
df_gb = pd.read_csv(gb_file)

# 2. Get ground truth consensus target
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# 3. Align data
df_eval = pd.merge(df_gb[['Player_Name', 'Maximal_ML_Rank']], df_master[['Player_Name', 'Consensus_Rank']], on='Player_Name')
df_eval['Rank_Delta'] = df_eval['Maximal_ML_Rank'] - df_eval['Consensus_Rank']

# 4. Calculate stats
mae = mean_absolute_error(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank'])
rmse = np.sqrt(mean_squared_error(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank']))
r2 = r2_score(df_eval['Consensus_Rank'], df_eval['Maximal_ML_Rank'])

print("🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬")
print(f"📈 Model R² Score (Variance Explained): {r2:.4f}")
print(f"🎯 Mean Absolute Error (Average Deviation): {mae:.2f} positions")
print(f"📉 Root Mean Squared Error (Outlier Penalty): {rmse:.2f} positions")
print("-" * 68)

# Show top 3 closest and top 3 furthest
print("\n🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nsmallest(3).index][['Player_Name', 'Consensus_Rank', 'Maximal_ML_Rank', 'Rank_Delta']].to_string(index=False))

print("\n⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nlargest(3).index][['Player_Name', 'Consensus_Rank', 'Maximal_ML_Rank', 'Rank_Delta']].to_string(index=False))
print("=====================================================================")

🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬
📈 Model R² Score (Variance Explained): 0.8155
🎯 Mean Absolute Error (Average Deviation): 11.64 positions
📉 Root Mean Squared Error (Outlier Penalty): 13.26 positions
--------------------------------------------------------------------

🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:
   Player_Name  Consensus_Rank  Maximal_ML_Rank  Rank_Delta
Michael Jordan        1.000000                1    0.000000
  LeBron James        2.000000                2    0.000000
 Dirk Nowitzki       18.333333               18   -0.333333

⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:
  Player_Name  Consensus_Rank  Maximal_ML_Rank  Rank_Delta
   Chris Bosh       82.333333               58  -24.333333
 Alex English       85.333333               63  -22.333333
Manu Ginobili       73.000000               52  -21.000000
